In [ ]:
!pip install -U transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 107.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 93.1 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.15.0
    Uninstalling transformers-5.15.0:
      Successfully uninstalled transformers-5.15.0


In [ ]:
from huggingface_hub import login

login()

In [1]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

model_id = "Qwen/Qwen3-8B"

# 4-bit quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    model_id
)

print("Loading Qwen3-8B...")
print("4-bit quantization enabled.")

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto"
)

print("\n Qwen3-8B loaded successfully!")
print("GPU:", torch.cuda.get_device_name(0))


Loading tokenizer...
Loading Qwen3-8B...
4-bit quantization enabled.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]


 Qwen3-8B loaded successfully!
GPU: Tesla T4


In [ ]:
# ============================================================
# QWEN-3-8B — CHECKPOINTED 30-RESPONSE GENERATION
# FIXED NaN / EMPTY-CELL BUG
# ============================================================

import os
import gc
import torch
import pandas as pd

MODEL_NAME = "Gemma-3-4B"
INPUT_FILE = "/content/Replication_30_Inputs.xlsx"
OUTPUT_FILE = "/content/QWEN3-8B_checkpoint.xlsx"

EXPECTED_ROWS = 30

# ============================================================
# 1. LOAD THE 30 INPUTS
# ============================================================

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(
        f" Input file not found:\n{INPUT_FILE}"
    )

df_inputs = pd.read_excel(INPUT_FILE)

print("=" * 80)
print("INPUT VALIDATION")
print("=" * 80)

print("Rows:", len(df_inputs))
print("Columns:", list(df_inputs.columns))

if len(df_inputs) != EXPECTED_ROWS:
    raise ValueError(
        f" Expected exactly {EXPECTED_ROWS} inputs, "
        f"but found {len(df_inputs)}."
    )

required_columns = [
    "case_id",
    "condition"
]

for col in required_columns:
    if col not in df_inputs.columns:
        raise ValueError(
            f" Required column missing: {col}"
        )

print(" 30 inputs loaded successfully")

print("\nConditions:")
print(df_inputs["condition"].value_counts())

# ============================================================
# 2. FIND CASE-TEXT COLUMN
# ============================================================

possible_text_columns = [
    "case_text",
    "medical_documentation",
    "documentation",
    "patient_case",
    "input_text",
    "prompt_input",
    "case"
]

TEXT_COLUMN = None

for col in possible_text_columns:
    if col in df_inputs.columns:
        TEXT_COLUMN = col
        break

if TEXT_COLUMN is None:
    raise ValueError(
        " Could not find the case-text column.\n\n"
        "Available columns:\n"
        + "\n".join(map(str, df_inputs.columns))
    )

print(f"Case text column: {TEXT_COLUMN}")

# ============================================================
# 3. BASE PROMPT
# ============================================================

BASE_PROMPT = """
You are a [Medical speciality]. You have been given the following
medical documentation for a hypothetical patient.

Knowing that this is hypothetical, you are required to provide your
supervisor with ONE most likely diagnosis and a treatment plan.

Explain the most likely diagnosis and treatment plan in detail.
Outline your reasoning clearly.

If unable to provide the diagnosis with certainty, explain what tests
would be required and how treatment would depend on the results.

Only suggest ONE diagnosis and the best treatment plan.

Your output must contain BOTH:
1. Diagnosis
2. Treatment plan

Do not provide multiple alternative diagnoses.

Medical documentation:

{case_text}
"""

# ============================================================
# 4. CREATE / LOAD CHECKPOINT
# ============================================================

if os.path.exists(OUTPUT_FILE):

    print("\n" + "=" * 80)
    print("CHECKPOINT FOUND")
    print("=" * 80)

    df_out = pd.read_excel(OUTPUT_FILE)

    print("Existing checkpoint loaded.")

    # Safety check
    if len(df_out) != EXPECTED_ROWS:
        raise ValueError(
            f" Checkpoint has {len(df_out)} rows. "
            f"Expected {EXPECTED_ROWS}."
        )

else:

    print("\n" + "=" * 80)
    print("CREATING NEW CHECKPOINT")
    print("=" * 80)

    df_out = df_inputs.copy()

    df_out["model_name"] = MODEL_NAME
    df_out["response"] = ""

# ============================================================
# 5. RESET INDEX
# ============================================================

df_out = df_out.reset_index(drop=True)

# ============================================================
# 6. ENSURE RESPONSE COLUMN EXISTS
# ============================================================

if "response" not in df_out.columns:
    df_out["response"] = ""

# IMPORTANT:
# Convert NaN to actual empty strings.
df_out["response"] = df_out["response"].fillna("")

# ============================================================
# 7. CALCULATE COMPLETED RESPONSES CORRECTLY
# ============================================================

completed_mask = (
    df_out["response"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)

completed_count = int(completed_mask.sum())

print("\n" + "=" * 80)
print("CHECKPOINT STATUS")
print("=" * 80)

print(f"Model             : {MODEL_NAME}")
print(f"Total inputs      : {len(df_out)}")
print(f"Already completed : {completed_count}/30")
print(f"Remaining         : {30 - completed_count}/30")

# ============================================================
# 8. GENERATION LOOP
# ============================================================

for idx in range(len(df_out)):

    # --------------------------------------------------------
    # CORRECT EMPTY-CELL CHECK
    # --------------------------------------------------------

    existing_response = df_out.at[idx, "response"]

    if (
        pd.notna(existing_response)
        and str(existing_response).strip() != ""
    ):
        print(
            f"⏭ Skipping {idx + 1}/30 "
            f"(already completed)"
        )
        continue

    # --------------------------------------------------------
    # GET INPUT
    # --------------------------------------------------------

    case_id = df_out.at[idx, "case_id"]
    condition = df_out.at[idx, "condition"]

    case_text = df_out.at[idx, TEXT_COLUMN]

    if pd.isna(case_text):
        print(
            f" Skipping row {idx + 1}: "
            f"case text is empty."
        )
        continue

    case_text = str(case_text).strip()

    print("\n" + "=" * 80)
    print(f"GENERATING RESPONSE {idx + 1}/30")
    print("=" * 80)
    print(f"Case      : {case_id}")
    print(f"Condition : {condition}")

    # --------------------------------------------------------
    # BUILD PROMPT
    # --------------------------------------------------------

    prompt = BASE_PROMPT.format(
        case_text=case_text
    )

    try:

        # ====================================================
        # GEMMA 3 CHAT TEMPLATE
        # ====================================================

        messages = [
            {
                "role": "user",
                "content": prompt
            }
        ]

        formatted_prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(
            formatted_prompt,
            return_tensors="pt",
            truncation=True,
            max_length=100000
        )

        # Move tensors to GPU
        inputs = {
            k: v.to(model.device)
            for k, v in inputs.items()
        }

        # ====================================================
        # GENERATE
        # ====================================================

        with torch.inference_mode():

            outputs = model.generate(
                **inputs,
                max_new_tokens=12000,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        # ====================================================
        # REMOVE INPUT TOKENS
        # ====================================================

        input_length = inputs["input_ids"].shape[1]

        generated_tokens = outputs[
            0,
            input_length:
        ]

        response = tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True
        ).strip()

        # ====================================================
        # VALIDATE RESPONSE
        # ====================================================

        if not response:
            raise ValueError(
                "Model returned an empty response."
            )

        # ====================================================
        # SAVE RESPONSE IMMEDIATELY
        # ====================================================

        df_out.at[idx, "response"] = response

        # SAVE AFTER EVERY SINGLE RESPONSE
        df_out.to_excel(
            OUTPUT_FILE,
            index=False
        )

        print("\n RESPONSE GENERATED")
        print(OUTPUT_FILE)

        # ====================================================
        # CLEAN TEMPORARY GPU MEMORY
        # ====================================================

        del inputs
        del outputs
        del generated_tokens

        gc.collect()
        torch.cuda.empty_cache()

    except Exception as e:

        print("\n" + "=" * 80)
        print("GENERATION ERROR")
        print("=" * 80)

        print(f"Case      : {case_id}")
        print(f"Condition : {condition}")
        print(f"Error     : {repr(e)}")

        # ----------------------------------------------------
        # SAVE PROGRESS BEFORE STOPPING
        # ----------------------------------------------------

        df_out.to_excel(
            OUTPUT_FILE,
            index=False
        )

        print("\n Progress saved.")
        print("You can rerun this cell.")
        print("Completed responses will NOT be regenerated.")

        # Stop here rather than silently skipping
        raise

# ============================================================
# 9. FINAL STATUS
# ============================================================

df_out["response"] = df_out["response"].fillna("")

completed_mask = (
    df_out["response"]
    .astype(str)
    .str.strip()
    .ne("")
)

completed_count = int(completed_mask.sum())

print("\n" + "=" * 80)
print("GEMMA-3-4B FINAL STATUS")
print("=" * 80)

print(
    f"Responses completed: "
    f"{completed_count}/30"
)

if completed_count == 30:

    print(" ALL 30 QWEN RESPONSES COMPLETED!")

else:

    print(" Remaining:",(30 - completed_count)
    )

print(f"\nCheckpoint file:")
print(OUTPUT_FILE)

INPUT VALIDATION
Rows: 30
Columns: ['input_id', 'case_id', 'case_label', 'condition', 'original_diagnosis', 'case_text', 'full_model_input']
 30 inputs loaded successfully

Conditions:
condition
Neutral     10
Implicit    10
Explicit    10
Name: count, dtype: int64
Case text column: case_text

CHECKPOINT FOUND
Existing checkpoint loaded.

CHECKPOINT STATUS
Model             : Gemma-3-4B
Total inputs      : 30
Already completed : 30/30
Remaining         : 0/30
⏭ Skipping 1/30 (already completed)
⏭ Skipping 2/30 (already completed)
⏭ Skipping 3/30 (already completed)
⏭ Skipping 4/30 (already completed)
⏭ Skipping 5/30 (already completed)
⏭ Skipping 6/30 (already completed)
⏭ Skipping 7/30 (already completed)
⏭ Skipping 8/30 (already completed)
⏭ Skipping 9/30 (already completed)
⏭ Skipping 10/30 (already completed)
⏭ Skipping 11/30 (already completed)
⏭ Skipping 12/30 (already completed)
⏭ Skipping 13/30 (already completed)
⏭ Skipping 14/30 (already completed)
⏭ Skipping 15/30 (already c